# Qurany Whisper fine-tune (Kaggle) — Surah Al-Baqarah

Fine-tunes [tarteel-ai/whisper-base-ar-quran](https://huggingface.co/tarteel-ai/whisper-base-ar-quran) on **all 286 ayahs of Surah 2 (Al-Baqarah)** with LoRA, keeping full **tashkeel/harakat** in the labels, and exports **`qurany-whisper-base-al-baqarah`**.

**Kaggle settings:** Internet ON · **GPU T4 x 2** (or P100 with `NUM_GPUS = 1`)

**Setup checklist (see `README.md` in this folder for the full path reference):**
1. Nothing to upload manually - Cell 2 below (`%%writefile`) writes `qurany_whisper_finetune.py` straight to `/kaggle/working/` when you run it, since the full script is embedded in this notebook.
2. *(Optional, for the mistake-detection demo)* **Add Data** -> create/attach a Kaggle Dataset whose slug is exactly `qurany-training-data`, containing (all at the dataset's **root**, not in a subfolder):
   - `Al-Baqarah.txt`
   - your full Al-Baqarah recitation (any reciter) + short test clips (`.mp3`/`.wav`/`.m4a`/`.ogg`)

   These are **never used for training** - only for a held-out WER + mistake-detection sanity check after training finishes. If you skip this, `cfg.external_test_dir` just won't exist and that step is silently skipped (no error).

Epochs are capped at 60 but `load_best_model_at_end` + early stopping on WER will normally stop the run well before that, picking whichever checkpoint generalizes best instead of the most-overfit one.


In [ ]:
!pip install -q -U transformers datasets peft accelerate evaluate jiwer soundfile librosa pandas torchaudio


In [ ]:
%%writefile /kaggle/working/qurany_whisper_finetune.py
"""
Qurany Whisper fine-tune for Kaggle (2x GPU via accelerate.notebook_launcher).

Base model : tarteel-ai/whisper-base-ar-quran
Data scope : Surah 2 (Al-Baqarah), all 286 ayahs — single reciter (EveryAyah.com)
Output     : /kaggle/working/qurany-whisper-base-al-baqarah (+ checkpoint zips)

Why Al-Baqarah only: it lets us push epochs (not just 3-4) within an 8h Kaggle
session while keeping the base model's general Quranic knowledge intact (LoRA,
small rank), and gives clean, single-reciter labels for a focused MVP demo.

Extras vs. a plain fine-tune:
  - Harakat (tashkeel) are preserved in the training labels (Uthmani script
    from AlQuran Cloud), so the model is trained on full diacritics, not just
    bare letters — this is what a Tajweed-aware ASR needs.
  - `load_best_model_at_end` + `EarlyStoppingCallback` let you set a *high*
    epoch budget (default 60) without babysitting the run: training stops
    automatically once WER stops improving on the held-out split, and the
    best-by-WER checkpoint (not necessarily the last one) is what gets saved,
    which also guards against overfitting 286 ayahs for 60 epochs.
  - `AutoDownloadCheckpoint` zips every saved checkpoint straight to
    /kaggle/working (+ /kaggle/output) with a FileLink, so a Kaggle disconnect
    never loses progress (pattern adapted from a teammate's Kaggle training
    script that zips + auto-links checkpoints on `on_save`).
  - Optional external test set: point `cfg.external_test_dir` at a Kaggle
    "Add Data" dataset holding your own recordings (e.g. a full Al-Baqarah
    recitation by a different reciter + a handful of short clips) and
    `run_external_evaluation()` will transcribe them with the fine-tuned model
    and run them through a self-contained port of the backend's
    Needleman-Wunsch mistake-detector, so you can sanity-check real-world
    mistake detection before wiring the model into the FastAPI backend.

Copy this file into a Kaggle notebook or run cells from Qurany_Whisper_Finetune.ipynb.
"""
from __future__ import annotations

import glob
import json
import os
import re
import shutil
import time
import urllib.request
from dataclasses import dataclass
from pathlib import Path

# ── Qurany branding (saved into the exported model card) ─────────────────────
QURANY_MODEL_NAME = "qurany-whisper-base-al-baqarah"
QURANY_MODEL_VERSION = "1.0.0"
QURANY_BASE_MODEL = "tarteel-ai/whisper-base-ar-quran"
QURANY_DESCRIPTION = (
    "Fine-tuned from Tarteel's whisper-base-ar-quran on all 286 ayahs of Surah "
    "Al-Baqarah for the Qurany graduation recitation assistant. Labels keep full "
    "Uthmani tashkeel (harakat) so the model stays diacritic-aware."
)

AL_BAQARAH_SURAH = 2

RECITER_FOLDER = "Alafasy_128kbps"
EVERYAYAH_URL = "https://everyayah.com/data/{reciter}/{s:03d}{a:03d}.mp3"
ALQURAN_API = "https://api.alquran.cloud/v1"

# Standard madani mushaf ayah counts per surah (1-indexed surah number).
SURAH_AYAH_COUNTS = [
    7, 286, 200, 176, 120, 165, 206, 75, 129, 109, 123, 111, 43, 52, 99, 128,
    111, 110, 98, 135, 112, 78, 118, 64, 77, 227, 93, 88, 69, 60, 34, 30, 73,
    54, 45, 83, 182, 88, 75, 85, 54, 53, 89, 59, 37, 35, 38, 29, 18, 45, 60,
    49, 62, 55, 78, 96, 29, 22, 24, 13, 14, 11, 11, 18, 12, 12, 30, 52, 52, 44,
    28, 28, 20, 56, 40, 31, 50, 40, 46, 42, 29, 19, 36, 25, 22, 17, 19, 26, 30,
    20, 15, 21, 11, 8, 8, 19, 5, 8, 8, 11, 11, 8, 3, 9, 5, 4, 7, 3, 6, 3, 5, 4,
    5, 6,
]


@dataclass
class TrainConfig:
    data_dir: Path = Path("/kaggle/working/quran_data")
    output_dir: Path = Path("/kaggle/working/runs")
    final_model_dir: Path = Path("/kaggle/working") / QURANY_MODEL_NAME
    kaggle_output: Path = Path("/kaggle/output")
    working_dir: Path = Path("/kaggle/working")

    # Point this at a Kaggle "Add Data" dataset holding your own recordings
    # (full Al-Baqarah recitation + short clips) for held-out evaluation only —
    # they are NEVER mixed into the training set, since we don't have a
    # verified per-ayah ground truth for them. Leave missing to skip.
    external_test_dir: Path | None = Path("/kaggle/input/qurany-training-data")

    reciter: str = RECITER_FOLDER
    surah_number: int = AL_BAQARAH_SURAH
    max_train_samples: int | None = None  # set for smoke tests

    num_train_epochs: float = 60
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 8
    gradient_accumulation_steps: int = 2
    learning_rate: float = 1e-5
    warmup_steps: int = 50
    save_steps: int = 50
    eval_steps: int = 50
    logging_steps: int = 10
    lora_r: int = 32
    lora_alpha: int = 64
    eval_holdout_ratio: float = 0.1
    seed: int = 42

    # `load_best_model_at_end` picks the lowest-WER checkpoint; early stopping
    # kills the run once WER stops improving, so "60 epochs" is a ceiling, not
    # a target — this is what keeps a 286-ayah dataset from just overfitting.
    early_stopping_patience: int = 8         # evals without WER improvement
    early_stopping_threshold: float = 0.05   # in WER percentage points


def _http_get_json(url: str, timeout: float = 60.0) -> dict:
    req = urllib.request.Request(url, headers={"User-Agent": "QuranyTrainer/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def _download_file(url: str, dest: Path, retries: int = 3) -> bool:
    if dest.exists() and dest.stat().st_size > 0:
        return True
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "QuranyTrainer/1.0"})
            with urllib.request.urlopen(req, timeout=120) as resp:
                dest.write_bytes(resp.read())
            return dest.stat().st_size > 0
        except Exception:
            time.sleep(1.5 * (attempt + 1))
    return False


def _fetch_ayah_text(surah: int, ayah: int, cache: dict[tuple[int, int], str]) -> str:
    key = (surah, ayah)
    if key in cache:
        return cache[key]
    if surah not in cache.get("_surah_loaded", {}):
        payload = _http_get_json(f"{ALQURAN_API}/surah/{surah}/ar.uthmani")
        ayahs = payload["data"]["ayahs"]
        for a in ayahs:
            cache[(surah, a["numberInSurah"])] = a["text"]
        cache["_surah_loaded"] = cache.get("_surah_loaded", {})
        cache["_surah_loaded"][surah] = True
    return cache[(surah, ayah)]


def iter_surah_ayahs(surah_number: int):
    """Yield (surah, ayah) for every ayah in a single surah."""
    total = SURAH_AYAH_COUNTS[surah_number - 1]
    for ayah in range(1, total + 1):
        yield surah_number, ayah


def download_training_data(cfg: TrainConfig) -> Path:
    """Download EveryAyah audio + Uthmani labels; write manifest.csv."""
    audio_root = cfg.data_dir / "audio"
    manifest_path = cfg.data_dir / "manifest.csv"
    cfg.data_dir.mkdir(parents=True, exist_ok=True)

    text_cache: dict = {}
    rows: list[dict] = []
    failed: list[str] = []

    pairs = list(iter_surah_ayahs(cfg.surah_number))
    if cfg.max_train_samples:
        pairs = pairs[: cfg.max_train_samples]

    print(f"Downloading {len(pairs)} ayah clips (Surah {cfg.surah_number}, reciter={cfg.reciter})...")

    for i, (surah, ayah) in enumerate(pairs, start=1):
        rel = f"{surah:03d}_{ayah:03d}.mp3"
        dest = audio_root / rel
        url = EVERYAYAH_URL.format(reciter=cfg.reciter, s=surah, a=ayah)
        if not _download_file(url, dest):
            failed.append(rel)
            continue
        try:
            text = _fetch_ayah_text(surah, ayah, text_cache).strip()
        except Exception:
            failed.append(rel)
            continue
        if not text:
            failed.append(rel)
            continue
        rows.append(
            {
                "audio_path": str(dest),
                "text": text,
                "surah": surah,
                "ayah": ayah,
            }
        )
        if i % 50 == 0:
            print(f"  … {i}/{len(pairs)} processed, {len(rows)} ok, {len(failed)} failed")

    import pandas as pd

    df = pd.DataFrame(rows)
    df.to_csv(manifest_path, index=False)
    meta = {
        "total_requested": len(pairs),
        "downloaded": len(rows),
        "failed": len(failed),
        "reciter": cfg.reciter,
        "scope": f"Surah {cfg.surah_number} (all ayahs)",
        "base_model": QURANY_BASE_MODEL,
        "qurany_model_name": QURANY_MODEL_NAME,
    }
    (cfg.data_dir / "download_meta.json").write_text(
        json.dumps(meta, indent=2), encoding="utf-8"
    )
    print(json.dumps(meta, indent=2))
    if len(rows) < 200:
        raise RuntimeError(
            f"Too few samples downloaded ({len(rows)}). Check Kaggle internet / EveryAyah URLs "
            f"(try a different RECITER_FOLDER, e.g. Husary_64kbps, if Alafasy_128kbps 404s)."
        )
    return manifest_path


def zip_and_link(
    source_path: Path,
    zip_basename: str,
    working_dir: Path,
    kaggle_output: Path,
) -> None:
    """Zip a file or directory and show Kaggle download links (main process only)."""
    try:
        from IPython.display import FileLink, display
    except ImportError:  # pragma: no cover - only missing outside Jupyter
        FileLink = None
        display = print

    working_dir.mkdir(parents=True, exist_ok=True)
    kaggle_output.mkdir(parents=True, exist_ok=True)

    zip_base = working_dir / zip_basename
    if source_path.is_dir():
        archive = shutil.make_archive(str(zip_base), "zip", root_dir=source_path)
    else:
        # single file: make a temp dir wrapper
        staging = working_dir / f"_stage_{zip_basename}"
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True)
        shutil.copy2(source_path, staging / source_path.name)
        archive = shutil.make_archive(str(zip_base), "zip", root_dir=staging)
        shutil.rmtree(staging)

    archive_path = Path(archive)
    out_copy = kaggle_output / archive_path.name
    shutil.copy2(archive_path, out_copy)

    print(f"Saved checkpoint archive: {archive_path} ({archive_path.stat().st_size / 1e6:.1f} MB)")
    if FileLink is not None:
        display(FileLink(str(archive_path)))
        display(FileLink(str(out_copy)))


class AutoDownloadCheckpoint:
    """Zip each Trainer checkpoint and show a Kaggle FileLink (main process only)."""

    def __init__(
        self,
        working_dir: Path | str = "/kaggle/working",
        kaggle_output: Path | str = "/kaggle/output",
    ) -> None:
        from transformers import TrainerCallback

        self.working_dir = Path(working_dir)
        self.kaggle_output = Path(kaggle_output)
        self._base = TrainerCallback

    def __call__(self):
        """Return a TrainerCallback subclass instance (HF expects instances in callbacks=)."""
        working_dir = self.working_dir
        kaggle_output = self.kaggle_output
        BaseCallback = self._base

        class _Callback(BaseCallback):
            def on_save(self, args, state, control, **kwargs):
                if not state.is_world_process_zero:
                    return control
                step = state.global_step
                ckpt_dir = Path(args.output_dir) / f"checkpoint-{step}"
                if not ckpt_dir.is_dir():
                    return control
                try:
                    zip_and_link(ckpt_dir, f"checkpoint{step}", working_dir, kaggle_output)
                except Exception as exc:
                    print(f"AutoDownloadCheckpoint @ step {step}: {exc}")
                return control

        return _Callback()


# ── Self-contained mistake-detector (ported from backend/src/services) ───────
# Kept dependency-free (no import of the FastAPI backend package) so it runs
# standalone inside a Kaggle kernel.

_DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670]")
_TATWEEL_CHAR = "\u0640"
_ALEF_VARIANTS_RE = re.compile(r"[\u0625\u0623\u0622\u0671\u0627]")
_NON_ARABIC_RE = re.compile(r"[^\u0621-\u064A\s]")
_MULTISPACE_RE = re.compile(r"\s+")


def _normalize_arabic(text: str) -> str:
    if not text:
        return ""
    text = _DIACRITICS_RE.sub("", text)
    text = text.replace(_TATWEEL_CHAR, "")
    text = _ALEF_VARIANTS_RE.sub("\u0627", text)
    text = text.replace("\u0624", "\u0648")
    text = text.replace("\u0626", "\u064A")
    text = text.replace("\u0649", "\u064A")
    text = text.replace("\u0629", "\u0647")
    text = text.replace("\u0621", "")
    text = _NON_ARABIC_RE.sub("", text)
    text = _MULTISPACE_RE.sub(" ", text).strip()
    return text


def _word_similarity(a: str, b: str) -> float:
    if a == b:
        return 1.0
    set_a, set_b = set(a), set(b)
    union = set_a | set_b
    return (len(set_a & set_b) / len(union)) if union else 0.0


def _needleman_wunsch_errors(reference_text: str, hyp_text: str, threshold: float = 0.6) -> dict:
    """Word-level Needleman-Wunsch alignment; mirrors alignment_service.py."""
    ref = [w for w in (_normalize_arabic(w) for w in reference_text.split()) if w]
    hyp = [w for w in (_normalize_arabic(w) for w in hyp_text.split()) if w]
    n, m = len(ref), len(hyp)

    def matches(a: str, b: str) -> bool:
        return a == b or _word_similarity(a, b) > threshold

    cost = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        cost[i][0] = i
    for j in range(1, m + 1):
        cost[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub = 0 if matches(ref[i - 1], hyp[j - 1]) else 1
            cost[i][j] = min(
                cost[i - 1][j - 1] + sub,
                cost[i - 1][j] + 1,
                cost[i][j - 1] + 1,
            )

    ops: list[tuple[str, str, str]] = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and cost[i][j] == cost[i - 1][j - 1] + (0 if matches(ref[i - 1], hyp[j - 1]) else 1):
            kind = "match" if matches(ref[i - 1], hyp[j - 1]) else "substitution"
            ops.append((kind, ref[i - 1], hyp[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and cost[i][j] == cost[i - 1][j] + 1:
            ops.append(("omission", ref[i - 1], ""))
            i -= 1
        else:
            ops.append(("addition", "", hyp[j - 1]))
            j -= 1
    ops.reverse()

    errors = [op for op in ops if op[0] != "match"]
    matched = n - sum(1 for op in errors if op[0] != "addition")
    similarity = max(0.0, min(100.0, (matched / n * 100) if n else 0.0))
    return {
        "total_words": n,
        "matched_words": max(matched, 0),
        "mistakes": len(errors),
        "similarity_score": round(similarity, 2),
        "errors": errors,
    }


def _parse_reference_txt(path: Path) -> dict[int, str]:
    """Parse the '<ayah text> (<n>) <ayah text> (<n>) ...' canonical-text format
    used by training data/Al-Baqarah.txt into {ayah_number: text}."""
    raw = path.read_text(encoding="utf-8")
    ayahs: dict[int, str] = {}
    for match in re.finditer(r"(.+?)\((\d+)\)", raw, flags=re.DOTALL):
        text, num = match.group(1).strip(), int(match.group(2))
        ayahs[num] = text
    return ayahs


def run_external_evaluation(cfg: TrainConfig, model, processor) -> None:
    """Transcribe the user's own recordings and demo mistake detection.

    Looks in `cfg.external_test_dir` for:
      - Al-Baqarah.txt (canonical text, "ayah text (n)" format) — optional,
        falls back to the downloaded training manifest's text.
      - one or more .mp3/.wav/.m4a/.ogg files: each is transcribed independently
        and diffed against the full Al-Baqarah text. If a clip only covers a
        few ayat, most of the surah will show as 'omission' — read this as a
        word-diff / mistake-detection smoke test, not a strict WER.
    """
    import librosa
    import pandas as pd
    import torch

    test_dir = cfg.external_test_dir
    if test_dir is None or not test_dir.exists():
        print(f"No external test set at {test_dir}; skipping external evaluation.")
        return

    txt_path = test_dir / "Al-Baqarah.txt"
    if txt_path.exists():
        ayah_map = _parse_reference_txt(txt_path)
        reference_text = " ".join(ayah_map[k] for k in sorted(ayah_map))
        print(f"Loaded canonical text from {txt_path} ({len(ayah_map)} ayahs).")
    else:
        manifest = pd.read_csv(cfg.data_dir / "manifest.csv")
        reference_text = " ".join(manifest.sort_values(["surah", "ayah"])["text"].tolist())
        print("No Al-Baqarah.txt in external test dir; using downloaded manifest text instead.")

    audio_files = sorted(
        {
            p
            for ext in ("*.mp3", "*.wav", "*.m4a", "*.ogg")
            for p in glob.glob(str(test_dir / ext))
        }
    )
    if not audio_files:
        print(f"No audio files found in {test_dir}; skipping external evaluation.")
        return

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()

    print(f"\nRunning external evaluation on {len(audio_files)} file(s) from {test_dir}...")
    for audio_path in audio_files:
        speech, _ = librosa.load(audio_path, sr=16000, mono=True)
        chunk_len = 16000 * 30  # Whisper decodes ~30s per forward pass
        transcript_parts: list[str] = []
        for start in range(0, max(len(speech), 1), chunk_len):
            chunk = speech[start : start + chunk_len]
            if chunk.size == 0:
                continue
            inputs = processor.feature_extractor(
                chunk, sampling_rate=16000, return_tensors="pt"
            ).input_features.to(device)
            with torch.no_grad():
                generated_ids = model.generate(inputs, max_new_tokens=225)
            transcript_parts.append(
                processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            )
        transcript = " ".join(transcript_parts).strip()

        report = _needleman_wunsch_errors(reference_text, transcript)
        print(f"\n=== {Path(audio_path).name} ===")
        print(
            f"  similarity: {report['similarity_score']}%  "
            f"matched: {report['matched_words']}/{report['total_words']}  "
            f"mistakes: {report['mistakes']}"
        )
        for kind, expected, recited in report["errors"][:15]:
            print(f"    [{kind}] expected='{expected}' recited='{recited}'")

        out_path = cfg.working_dir / f"external_eval_{Path(audio_path).stem}.json"
        out_path.write_text(
            json.dumps(
                {"audio": str(audio_path), "transcript": transcript, **report},
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        print(f"  -> saved {out_path}")


def _write_qurany_model_card(model_dir: Path, cfg: TrainConfig) -> None:
    card = f"""---
language: ar
license: apache-2.0
base_model: {QURANY_BASE_MODEL}
tags:
  - whisper
  - quran
  - qurany
  - automatic-speech-recognition
library_name: transformers
pipeline_tag: automatic-speech-recognition
---

# {QURANY_MODEL_NAME}

**Version:** {QURANY_MODEL_VERSION}

{QURANY_DESCRIPTION}

## Base model

Fine-tuned from [{QURANY_BASE_MODEL}](https://huggingface.co/{QURANY_BASE_MODEL}) (Tarteel AI).

## Training data

- **Scope:** Surah {cfg.surah_number} (Al-Baqarah) — all 286 ayahs
- **Reciter:** {cfg.reciter} (EveryAyah.com)
- **Labels:** Uthmani script with full tashkeel/harakat (AlQuran Cloud API)
- **Epoch budget:** up to {cfg.num_train_epochs}, `load_best_model_at_end` on WER +
  early stopping (patience={cfg.early_stopping_patience}) to avoid overfitting a
  single-surah dataset.

## Intended use

Speech-to-text for Quranic Arabic recitation in the **Qurany** mobile app backend.
Downstream word alignment and mistake detection are handled separately
(see `backend/src/services/alignment_service.py`).

## Team

Cairo University — Faculty of Computing and Artificial Intelligence (Graduation Project 2025–2026)
"""
    model_dir.mkdir(parents=True, exist_ok=True)
    (model_dir / "README.md").write_text(card, encoding="utf-8")
    (model_dir / "qurany_model_info.json").write_text(
        json.dumps(
            {
                "name": QURANY_MODEL_NAME,
                "version": QURANY_MODEL_VERSION,
                "base_model": QURANY_BASE_MODEL,
                "training_scope": f"Surah {cfg.surah_number} (Al-Baqarah)",
            },
            indent=2,
        ),
        encoding="utf-8",
    )


def train_qurany_whisper(cfg: TrainConfig | None = None) -> None:
    """Main training entry (call inside accelerate notebook_launcher worker)."""
    import librosa
    import pandas as pd
    import torch
    from datasets import Dataset, DatasetDict
    from peft import LoraConfig, get_peft_model
    from transformers import (
        EarlyStoppingCallback,
        Seq2SeqTrainer,
        Seq2SeqTrainingArguments,
        WhisperForConditionalGeneration,
        WhisperProcessor,
        WhisperTokenizer,
    )

    cfg = cfg or TrainConfig()
    cfg.kaggle_output.mkdir(parents=True, exist_ok=True)
    cfg.output_dir.mkdir(parents=True, exist_ok=True)

    manifest = cfg.data_dir / "manifest.csv"
    if not manifest.exists():
        raise FileNotFoundError(
            f"{manifest} not found. Run download_training_data() in the notebook first."
        )

    df = pd.read_csv(manifest)
    df = df.sample(frac=1.0, random_state=cfg.seed).reset_index(drop=True)
    split_idx = int(len(df) * (1.0 - cfg.eval_holdout_ratio))
    ds = DatasetDict(
        {
            "train": Dataset.from_pandas(df.iloc[:split_idx]),
            "test": Dataset.from_pandas(df.iloc[split_idx:]),
        }
    )

    processor = WhisperProcessor.from_pretrained(QURANY_BASE_MODEL, language="ar", task="transcribe")
    tokenizer = WhisperTokenizer.from_pretrained(QURANY_BASE_MODEL, language="ar", task="transcribe")

    model = WhisperForConditionalGeneration.from_pretrained(QURANY_BASE_MODEL)
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []
    model.config.use_cache = False

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    def prepare_example(batch):
        audio_path = batch["audio_path"]
        speech, _ = librosa.load(audio_path, sr=16000, mono=True)
        batch["input_features"] = processor.feature_extractor(
            speech, sampling_rate=16000
        ).input_features[0]
        # NOTE: batch["text"] keeps full Uthmani tashkeel (harakat) — do NOT
        # normalize/strip diacritics here, that's what makes this a
        # harakat-aware fine-tune instead of a bare-letters one.
        batch["labels"] = tokenizer(batch["text"]).input_ids
        return batch

    ds = ds.map(
        prepare_example,
        remove_columns=ds["train"].column_names,
        num_proc=1,
    )

    class DataCollatorSpeechSeq2Seq:
        def __init__(self, processor, decoder_start_token_id: int):
            self.processor = processor
            self.decoder_start_token_id = decoder_start_token_id

        def __call__(self, features):
            input_features = [{"input_features": f["input_features"]} for f in features]
            batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
            label_features = [{"input_ids": f["labels"]} for f in features]
            labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
            labels = labels_batch["input_ids"].masked_fill(
                labels_batch.attention_mask.ne(1), -100
            )
            if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
                labels = labels[:, 1:]
            batch["labels"] = labels
            return batch

    data_collator = DataCollatorSpeechSeq2Seq(processor, model.config.decoder_start_token_id)

    n_gpus = max(1, torch.cuda.device_count())
    effective_batch = (
        cfg.per_device_train_batch_size * n_gpus * cfg.gradient_accumulation_steps
    )
    steps_per_epoch = max(1, len(ds["train"]) // effective_batch)
    total_steps = int(steps_per_epoch * cfg.num_train_epochs)
    print(
        f"GPUs={n_gpus}, train_samples={len(ds['train'])}, "
        f"effective_batch={effective_batch}, est_steps≈{total_steps} "
        f"(ceiling of {cfg.num_train_epochs} epochs; early stopping may cut this short)"
    )

    ta_kwargs = dict(
        output_dir=str(cfg.output_dir),
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        per_device_eval_batch_size=cfg.per_device_eval_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        warmup_steps=cfg.warmup_steps,
        num_train_epochs=cfg.num_train_epochs,
        fp16=torch.cuda.is_available(),
        gradient_checkpointing=True,
        save_strategy="steps",
        save_steps=cfg.save_steps,
        logging_steps=cfg.logging_steps,
        predict_with_generate=True,
        generation_max_length=225,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        save_total_limit=4,
        remove_unused_columns=False,
        label_names=["labels"],
        dataloader_num_workers=2,
        report_to=[],
        seed=cfg.seed,
        ddp_find_unused_parameters=False,
    )
    # Transformers renamed evaluation_strategy → eval_strategy in v4.41+
    try:
        training_args = Seq2SeqTrainingArguments(
            **ta_kwargs,
            eval_strategy="steps",
            eval_steps=cfg.eval_steps,
        )
    except TypeError:
        training_args = Seq2SeqTrainingArguments(
            **ta_kwargs,
            evaluation_strategy="steps",
            evaluation_steps=cfg.eval_steps,
        )

    try:
        import jiwer

        def compute_metrics(pred):
            pred_ids = pred.predictions
            label_ids = pred.label_ids
            label_ids[label_ids == -100] = tokenizer.pad_token_id
            pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
            label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
            wer = jiwer.wer(label_str, pred_str)
            return {"wer": wer * 100}

    except ImportError:
        compute_metrics = None

    trainer_kwargs = dict(
        args=training_args,
        model=model,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            AutoDownloadCheckpoint(cfg.working_dir, cfg.kaggle_output)(),
            EarlyStoppingCallback(
                early_stopping_patience=cfg.early_stopping_patience,
                early_stopping_threshold=cfg.early_stopping_threshold,
            ),
        ],
    )
    import inspect

    sig = inspect.signature(Seq2SeqTrainer.__init__)
    if "processing_class" in sig.parameters:
        trainer_kwargs["processing_class"] = processor
    else:
        trainer_kwargs["tokenizer"] = processor

    trainer = Seq2SeqTrainer(**trainer_kwargs)

    trainer.train()

    # Save Qurany-branded final model (merge LoRA for easier inference).
    is_main = int(os.environ.get("LOCAL_RANK", os.environ.get("RANK", "0"))) == 0
    if is_main:
        merged = trainer.model.merge_and_unload()
        cfg.final_model_dir.mkdir(parents=True, exist_ok=True)
        merged.save_pretrained(str(cfg.final_model_dir))
        processor.save_pretrained(str(cfg.final_model_dir))
        _write_qurany_model_card(cfg.final_model_dir, cfg)

        zip_and_link(
            cfg.final_model_dir,
            QURANY_MODEL_NAME,
            cfg.working_dir,
            cfg.kaggle_output,
        )
        print(f"Done. Final model: {cfg.final_model_dir}")

        try:
            run_external_evaluation(cfg, merged, processor)
        except Exception as exc:
            print(f"External evaluation skipped/failed: {exc}")


def launch_on_kaggle(num_processes: int = 2) -> None:
    """Download data then train on 2 GPUs using accelerate notebook_launcher."""
    from accelerate import notebook_launcher

    cfg = TrainConfig()
    download_training_data(cfg)

    print(f"Launching training on {num_processes} GPU(s)…")
    notebook_launcher(
        train_qurany_whisper,
        args=(cfg,),
        num_processes=num_processes,
    )


if __name__ == "__main__":
    # Local smoke test (CPU/single GPU): python qurany_whisper_finetune.py
    cfg = TrainConfig(max_train_samples=20, num_train_epochs=1, save_steps=5, eval_steps=5)
    download_training_data(cfg)
    train_qurany_whisper(cfg)


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

from qurany_whisper_finetune import TrainConfig, download_training_data, train_qurany_whisper

import torch
print("CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")


In [ ]:
cfg = TrainConfig()
# cfg.external_test_dir defaults to /kaggle/input/qurany-training-data -- change
# the folder name here if you named your "Add Data" dataset differently, or
# set it to None to skip external evaluation entirely.
print("External test dir:", cfg.external_test_dir, "exists=", cfg.external_test_dir and cfg.external_test_dir.exists())

manifest = download_training_data(cfg)
print("Manifest:", manifest)


In [ ]:
from accelerate import notebook_launcher

NUM_GPUS = 2  # T4 x2; use 1 for P100

# Trains up to cfg.num_train_epochs (60) with early stopping on WER, then
# saves the merged model, zips it, and (if cfg.external_test_dir exists) runs
# it against your own recordings for a mistake-detection sanity check.
notebook_launcher(
    train_qurany_whisper,
    args=(cfg,),
    num_processes=NUM_GPUS,
)


In [ ]:
import glob
import json

for path in sorted(glob.glob("/kaggle/working/external_eval_*.json")):
    data = json.load(open(path, encoding="utf-8"))
    print(f"\n{path}")
    print(f"  similarity={data['similarity_score']}%  matched={data['matched_words']}/{data['total_words']}  mistakes={data['mistakes']}")
